<a href="https://colab.research.google.com/github/mohatamegha/Gen-AI-Fundamentals/blob/main/langchain_retievers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install langchain chromadb faiss-cpu openai tiktoken langchain_openai langchain-community wikipedia

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 66.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 69.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.6/99.6 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 56.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 80.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 37.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 55.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━

In [2]:
# To securely use your API key, especially in Colab, you can store it in Colab's 'Secrets' tab.
# Then, you can access it like this:
import os
from google.colab import userdata

# Set the API key as an environment variable
# It's recommended to store your API key in Colab Secrets and access it via userdata.get()
# For example, if you saved your key as 'GOOGLE_API_KEY':
os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")


1. Wikipedia Retriever

In [19]:
# The `JSONDecodeError` you are seeing is happening when the `WikipediaRetriever`
# attempts to parse the response from the Wikipedia API in a *later* cell (LBwp9QvizIj7).
# This error usually indicates that the Wikipedia API returned something unexpected,
# such as an empty response, an HTML error page, or malformed data, which Python's
# JSON parser cannot interpret.
# The import statement in this cell (`5sS-fQMUzA6L`) is correct and not the cause of the error.
# To handle such issues robustly, a `try-except` block for `requests.exceptions.JSONDecodeError`
# should be added around the `retriever.invoke(query)` call in cell `LBwp9QvizIj7`.
from langchain_community.retrievers import WikipediaRetriever

In [4]:
# Initialize the retriever (optional: set language and top_k)
retriever = WikipediaRetriever(top_k_results=2, lang="en")

In [13]:
# Define your query
query = "the geopolitical history of india and pakistan from the perspective of a chinese"

# Get relevant Wikipedia documents
docs = retriever.invoke(query)

In [21]:
import requests

# Define your query
query2 = "the evolution of bollywood and indian cinema"

# Get relevant Wikipedia documents
try:
    # Corrected to use query2 instead of query
    docs2 = retriever.invoke(query2)
except requests.exceptions.JSONDecodeError as e:
    print(f"An error occurred while fetching documents from Wikipedia: {e}")
    print("This might be a temporary issue with the Wikipedia API or a network problem. Please try re-running the cell.")
    docs2 = [] # Initialize docs2 as an empty list to avoid further errors if the retrieval failed


In [22]:
# Print retrieved content
for i, doc in enumerate(docs2):
    print(f"\n--- Result {i+1} ---")
    print(f"Content:\n{doc.page_content}...")  # truncate for display


--- Result 1 ---
Content:
Rajesh Khanna (pronounced [ɾɑːd͡ʒeːʃ kʰənnɑː] ; born Jatin Khanna; 29 December 1942 – 18 July 2012) was an Indian actor, film producer and politician who worked in Hindi films. Regarded as one of the greatest and most successful actors in the history of Indian cinema, he is considered the first Superstar of Hindi cinema. His accolades include five Filmfare Awards, and in 2013, he was posthumously awarded the Padma Bhushan, India's third highest civilian honour.
Khanna made his acting debut in 1966 with Aakhri Khat, which was India's first official Academy Awards entry in 1967. 
Khanna saw his rise to superstardom in 1969  with romantic musical Aradhana starring Sharmila Tagore. The film went to become a massive blockbuster at the box office and made him an overnight sensation. The same year, romantic family drama Do Raaste opposite Mumtaz also released.  These films turned Khanna into a Superstar and marked the beginning of Rajesh Khanna Mania of the early-19

2. Vector Store Retriever

In [23]:
pip install langchain_google_genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.8/68.8 kB 2.6 MB/s eta 0:00:00


In [24]:
from langchain_community.vectorstores import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_core.documents import Document

In [66]:
documents = [
    Document(page_content="LangChain helps developers build LLM applications easily.", metadata={"source": "doc1"}),
    Document(page_content="Chroma is a vector database optimized for LLM-based search.", metadata={"source": "doc2"}),
    Document(page_content="Chroma is not the only vector database used for LLM-based apps.", metadata={"source": "doc3"}),
    Document(page_content="Embeddings convert text into high-dimensional vectors.", metadata={"source": "doc4"}),
    Document(page_content="OpenAI provides powerful embedding models.", metadata={"source": "doc5"}),
    Document(page_content="GeminiAI provides accurate embedding models.", metadata={"source": "doc6"}),
]

In [67]:
# Step 2: Initialize embedding model
embedding_model = GoogleGenerativeAIEmbeddings(model=("gemini-embedding-2"))

# Step 3: Create Chroma vector store in memory
vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embedding_model,
    collection_name="my_collection"
)

In [68]:
# Step 4: Convert vectorstore into a retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

In [69]:
query = "What is Chroma used for?"
results = retriever.invoke(query)

In [70]:
for i, doc in enumerate(results):
    print(f"\n--- Result {i+1} ---")
    print(f"Content: {doc.page_content}")
    print(f"Metadata: {doc.metadata}")


--- Result 1 ---
Content: Chroma is a vector database optimized for LLM-based search.
Metadata: {'source': 'doc2'}

--- Result 2 ---
Content: Chroma is not the only vector database used for LLM-based apps.
Metadata: {'source': 'doc3'}


### Clearing the Chroma Database

To clear the Chroma database, you can use the `delete_collection()` method on your `vectorstore` object. This will remove all documents and their associated embeddings from the specified collection.

In [65]:

print(f"Number of documents before clearing: {vectorstore._collection.count()}")

vectorstore.delete_collection()

NotFoundError: Collection [786edb49-b990-4e20-be54-5bf599290e5f] does not exist.

In [71]:
results = vectorstore.similarity_search(query, k=2)

In [72]:
for i, doc in enumerate(results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)


--- Result 1 ---
Chroma is a vector database optimized for LLM-based search.

--- Result 2 ---
Chroma is not the only vector database used for LLM-based apps.


3. MMR

In [73]:
# Sample documents
docs = [
    Document(page_content="LangChain makes it easy to work with LLMs."),
    Document(page_content="LangChain is used to build LLM based applications."),
    Document(page_content="Chroma is used to store and search document embeddings."),
    Document(page_content="Embeddings are vector representations of text."),
    Document(page_content="MMR helps you get diverse results when doing similarity search."),
    Document(page_content="LangChain supports Chroma, FAISS, Pinecone, and more."),
]

In [74]:
from langchain_community.vectorstores import FAISS

# Initialize OpenAI embeddings
embedding_model = GoogleGenerativeAIEmbeddings(model=("gemini-embedding-2"))

# Step 2: Create the FAISS vector store from documents
vectorstore = FAISS.from_documents(
    documents=docs,
    embedding=embedding_model
)

In [78]:
# Enable MMR in the retriever
retriever = vectorstore.as_retriever(
    search_type="mmr",                   # <-- This enables MMR
    search_kwargs={"k": 3, "lambda_mult": 1}  # k = top results, lambda_mult = relevance-diversity balance
)

In [79]:
query = "What is langchain?"
results = retriever.invoke(query)

In [80]:
for i, doc in enumerate(results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)


--- Result 1 ---
LangChain is used to build LLM based applications.

--- Result 2 ---
LangChain makes it easy to work with LLMs.

--- Result 3 ---
LangChain supports Chroma, FAISS, Pinecone, and more.


In [81]:
# Enable MMR in the retriever
retriever = vectorstore.as_retriever(
    search_type="mmr",                   # <-- This enables MMR
    search_kwargs={"k": 3, "lambda_mult": 0.5}  # k = top results, lambda_mult = relevance-diversity balance
)

In [82]:
query = "What is langchain?"
results = retriever.invoke(query)

In [83]:
for i, doc in enumerate(results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)


--- Result 1 ---
LangChain is used to build LLM based applications.

--- Result 2 ---
MMR helps you get diverse results when doing similarity search.

--- Result 3 ---
Chroma is used to store and search document embeddings.


4. Multiquery Retriever

In [92]:
pip install langchain-classic

In [107]:
from langchain_community.vectorstores import FAISS
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_core.documents import Document
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_classic.retrievers import MultiQueryRetriever

In [108]:
from langchain_google_genai import ChatGoogleGenerativeAI

In [109]:
model = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite")

In [110]:
model.invoke("HI")

AIMessage(content=[{'type': 'text', 'text': 'Hello! How can I help you today?', 'extras': {'signature': 'EjQKMgEMOdbHeY4TF/HBjUVJHjbA0ERJb7xSvWAb4ZRiMpE6vNOL16WlZD7ml3gDyIs2J5HH'}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019eab1b-b6f2-7402-ad23-670982310dce-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 2, 'output_tokens': 9, 'total_tokens': 11, 'input_token_details': {'cache_read': 0}})

In [111]:
# Relevant health & wellness documents
all_docs = [
    Document(page_content="Regular walking boosts heart health and can reduce symptoms of depression.", metadata={"source": "H1"}),
    Document(page_content="Consuming leafy greens and fruits helps detox the body and improve longevity.", metadata={"source": "H2"}),
    Document(page_content="Deep sleep is crucial for cellular repair and emotional regulation.", metadata={"source": "H3"}),
    Document(page_content="Mindfulness and controlled breathing lower cortisol and improve mental clarity.", metadata={"source": "H4"}),
    Document(page_content="Drinking sufficient water throughout the day helps maintain metabolism and energy.", metadata={"source": "H5"}),
    Document(page_content="The solar energy system in modern homes helps balance electricity demand.", metadata={"source": "I1"}),
    Document(page_content="Python balances readability with power, making it a popular system design language.", metadata={"source": "I2"}),
    Document(page_content="Photosynthesis enables plants to produce energy by converting sunlight.", metadata={"source": "I3"}),
    Document(page_content="The 2022 FIFA World Cup was held in Qatar and drew global energy and excitement.", metadata={"source": "I4"}),
    Document(page_content="Black holes bend spacetime and store immense gravitational energy.", metadata={"source": "I5"}),
]

In [112]:
# Initialize OpenAI embeddings
embedding_model = GoogleGenerativeAIEmbeddings(model=("gemini-embedding-2"))

# Create FAISS vector store
vectorstore = FAISS.from_documents(documents=all_docs, embedding=embedding_model)

In [113]:
# Create retrievers(Normal similarity search and multi query retriever)
similarity_retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 5})

In [114]:
multiquery_retriever = MultiQueryRetriever.from_llm(
    retriever=vectorstore.as_retriever(search_kwargs={"k": 5}),
    llm=model
)

In [115]:
# Query
query = "How to improve energy levels and maintain balance?"

In [116]:
# Retrieve results
similarity_results = similarity_retriever.invoke(query)
multiquery_results= multiquery_retriever.invoke(query)

In [117]:
for i, doc in enumerate(similarity_results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)

print("*"*150)

for i, doc in enumerate(multiquery_results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)


--- Result 1 ---
Drinking sufficient water throughout the day helps maintain metabolism and energy.

--- Result 2 ---
Mindfulness and controlled breathing lower cortisol and improve mental clarity.

--- Result 3 ---
Consuming leafy greens and fruits helps detox the body and improve longevity.

--- Result 4 ---
Deep sleep is crucial for cellular repair and emotional regulation.

--- Result 5 ---
Regular walking boosts heart health and can reduce symptoms of depression.
******************************************************************************************************************************************************

--- Result 1 ---
Drinking sufficient water throughout the day helps maintain metabolism and energy.

--- Result 2 ---
Mindfulness and controlled breathing lower cortisol and improve mental clarity.

--- Result 3 ---
Deep sleep is crucial for cellular repair and emotional regulation.

--- Result 4 ---
Regular walking boosts heart health and can reduce symptoms of depressio

ContextualCompressionRetriever

In [123]:
pip install langchain-classic

In [124]:
from langchain_community.vectorstores import FAISS
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor

# from langchain.retrievers import ContextualCompressionRetriever
# from langchain.retrievers.document_compressors import LLMChainExtractor
from langchain_core.documents import Document

In [126]:
from langchain_core.documents import Document

# Define the documents for compression
docs = [
    Document(page_content=(
        """The Grand Canyon is one of the most visited natural wonders in the world.
        Photosynthesis is the process by which green plants convert sunlight into energy.
        Millions of tourists travel to see it every year. The rocks date back millions of years."""
    ), metadata={"source": "Doc1"}),

    Document(page_content=(
        """In medieval Europe, castles were built primarily for defense.
        The chlorophyll in plant cells captures sunlight during photosynthesis.
        Knights wore armor made of metal. Siege weapons were often used to breach castle walls."""
    ), metadata={"source": "Doc2"}),

    Document(page_content=(
        """Basketball was invented by Dr. James Naismith in the late 19th century.
        It was originally played with a soccer ball and peach baskets. NBA is now a global league."""
    ), metadata={"source": "Doc3"}),

    Document(page_content=(
        """The history of cinema began in the late 1800s. Silent films were the earliest form.
        Thomas Edison was among the pioneers. Photosynthesis does not occur in animal cells.
        Modern filmmaking involves complex CGI and sound design."""
    ), metadata={"source": "Doc4"})
]

In [127]:
# Create a FAISS vector store from the documents
embedding_model = GoogleGenerativeAIEmbeddings(model=("gemini-embedding-2"))
vectorstore = FAISS.from_documents(docs, embedding_model)

In [128]:
base_retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

In [129]:
# Set up the compressor using an LLM
llm = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite")
compressor = LLMChainExtractor.from_llm(llm)

In [130]:
# Create the contextual compression retriever
compression_retriever = ContextualCompressionRetriever(
    base_retriever=base_retriever,
    base_compressor=compressor
)

In [131]:
# Query the retriever
query = "What is photosynthesis?"
compressed_results = compression_retriever.invoke(query)

In [132]:
for i, doc in enumerate(compressed_results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)



--- Result 1 ---
Photosynthesis is the process by which green plants convert sunlight into energy.

--- Result 2 ---
Extracted relevant parts: Photosynthesis does not occur in animal cells.

--- Result 3 ---
Extracted relevant parts: The chlorophyll in plant cells captures sunlight during photosynthesis.
